# RIFT-LoRA: Qwen 3B Competitor Matrix on Kaggle T4x2

This notebook clones VASTLoRA, installs the scale runner, and compares RIFT-LoRA against the strongest competitors that can be run honestly in the current 3B PEFT/QLoRA async simulator.

Main table metrics: `Accuracy`, `Loss`, `Harmful`, and `Late harmful`.

Main slices: `SST-2/IID`, `SST-2/non-IID + high staleness`, `QNLI/IID`, and `QNLI/non-IID + high staleness`.

Recommended mode for the thesis claim is `focused`: it runs SST-2 and QNLI only on the hard `non-IID + high staleness` slice with larger calibration/monitor splits and more measured returns. IID is useful as a sanity slice, but RIFT is not expected to dominate there because the late-update problem is much weaker.

Runnable methods in this notebook:

- `raw` / `fedex`: exact intrinsic LoRA innovation aggregation in the matched async simulator.
- `freshness`: scalar staleness decay.
- `fedrot`: FedRot-LoRA-style orthogonal Procrustes factor alignment.
- `vast`, `mtip`, `mtip_adaptive`: previous in-house transport baselines.
- `spectral_filter`: Spectral-Surgery-style gradient component filter, not the official paper implementation.
- `alignfed_calibration`: whole-update calibration gate, not full AlignFed.
- `rift`: proposed rank-wise objective filter plus paired calibration gate.

Skipped as full competitors: AdaLoRA, full Spectral Surgery, FLoRG, full GLoRA, full FedSteer, full AlignFed, and full OrthoFL. They are either not server aggregation methods, require a different LoRA parameterization/protocol, or do not have a trustworthy official integration path for this simulator. The analyzer writes those reasons into `skipped_competitors.json`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/TrgPhan/VASTLoRA.git"
REPO_REF = "b3de835cdbb8ca5e5fc459c11d02e6c6f6ec9bf1"
RUN_MODE = "focused"  # "pilot", "focused", or "full".

WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "VASTLoRA-rift-3b-run"
RESULT_DIR = WORK_ROOT / "rift-3b-competitor-results"
ANALYSIS_DIR = RESULT_DIR / "analysis"
assert RUN_MODE in {"pilot", "focused", "full"}
print({"repo_ref": REPO_REF, "run_mode": RUN_MODE})

## 1. Clone and install

In [ ]:
if REPO_DIR.exists():
    assert REPO_DIR.parent == WORK_ROOT
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[scale]"],
    check=True,
)
resolved_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print("Installed commit:", resolved_commit)

## 2. Verify GPUs and cache model/data

In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=True)
gpu_count = torch.cuda.device_count()
assert gpu_count >= 2, f"Expected Kaggle T4x2, found {gpu_count} CUDA device(s)"
print("CUDA devices:", [torch.cuda.get_device_name(i) for i in range(gpu_count)])

In [ ]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
snapshot_download(MODEL_NAME)
load_dataset("nyu-mll/glue", "sst2")
load_dataset("nyu-mll/glue", "qnli")
print("Model, SST-2, and QNLI are cached.")

## 3. Record official source availability

This cell clones official repos that are safe to audit without forcing them into an incompatible simulator. The actual runnable methods remain the matched implementations in this repo.

In [ ]:
SOURCE_DIR = WORK_ROOT / "rift-3b-official-sources"
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
official_repos = {
    "AdaLoRA": "https://github.com/QingruZhang/AdaLoRA.git",
    "FedRot-LoRA": "https://github.com/haoran-zh/FedRot-LoRA.git",
    "FedSteer": "https://github.com/haoran-zh/FedSteer.git",
}
cloned = {}
for name, url in official_repos.items():
    path = SOURCE_DIR / name
    if path.exists():
        shutil.rmtree(path)
    try:
        subprocess.run(["git", "clone", "--depth", "1", url, str(path)], check=True)
        cloned[name] = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=path, text=True).strip()
    except Exception as exc:
        cloned[name] = f"clone failed: {exc}"
(SOURCE_DIR / "source_commits.json").write_text(json.dumps(cloned, indent=2), encoding="utf-8")
cloned

## 4. Prepare run configuration

In [ ]:
BASE_CONFIG_PATH = REPO_DIR / "configs/kaggle_3b_rift_competitors.json"
RUNNER = REPO_DIR / "scripts/run_kaggle_3b.py"
ANALYZER = REPO_DIR / "scripts/analyze_kaggle_3b_rift_competitors.py"

base_config = json.loads(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
if RESULT_DIR.exists():
    assert RESULT_DIR.parent == WORK_ROOT
    shutil.rmtree(RESULT_DIR)
RESULT_DIR.mkdir(parents=True)
CONFIG_PATH = RESULT_DIR / "run_config.json"
LOG_DIR = RESULT_DIR / "logs"
LOG_DIR.mkdir()

task_specs = base_config["task_matrix"]["tasks"]
regime_specs = base_config["task_matrix"]["regimes"]
if RUN_MODE == "pilot":
    methods = ["raw", "freshness", "fedrot", "alignfed_calibration", "rift"]
    seeds = [4101]
    task_specs = task_specs
    regime_specs = regime_specs
    base_config["dataset"]["eval_examples"] = 128
    base_config["experiment"]["collected_returns"] = 8
    base_config["experiment"]["warmup_returns"] = 4
    base_config["experiment"]["calibration_gradient_examples"] = 8
    base_config["experiment"]["calibration_gate_examples"] = 16
    base_config["experiment"]["monitor_examples"] = 16
elif RUN_MODE == "focused":
    methods = ["raw", "fedex", "freshness", "fedrot", "spectral_filter", "alignfed_calibration", "rift"]
    seeds = base_config["experiment"]["seeds"]
    regime_specs = [item for item in regime_specs if item["name"] == "noniid_high_staleness"]
    base_config["dataset"]["eval_examples"] = 256
    base_config["experiment"]["warmup_returns"] = 8
    base_config["experiment"]["collected_returns"] = 32
    base_config["experiment"]["calibration_gradient_examples"] = 32
    base_config["experiment"]["calibration_gate_examples"] = 64
    base_config["experiment"]["monitor_examples"] = 64
else:
    methods = base_config["experiment"]["methods"]
    seeds = base_config["experiment"]["seeds"]

CONFIG_DIR = RESULT_DIR / "configs"
CONFIG_DIR.mkdir()

def write_matrix_config(task_spec, regime_spec):
    config = json.loads(json.dumps(base_config))
    config["output_dir"] = str(RESULT_DIR)
    config["dataset"].update(task_spec)
    config["experiment"]["methods"] = methods
    config["experiment"]["regime_name"] = regime_spec["name"]
    config["experiment"]["partition_mode"] = regime_spec["partition_mode"]
    config["experiment"]["client_ranks"] = regime_spec["client_ranks"]
    config["experiment"]["compute_times"] = regime_spec["compute_times"]
    path = CONFIG_DIR / f"{task_spec['name']}_{regime_spec['name']}.json"
    path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    return path

matrix_configs = {
    (task["name"], regime["name"]): write_matrix_config(task, regime)
    for task in task_specs
    for regime in regime_specs
}
print({"tasks": [t["name"] for t in task_specs], "regimes": [r["name"] for r in regime_specs], "methods": methods, "seeds": seeds})

for (task_name, regime_name), config_path in matrix_configs.items():
    for method in methods:
        subprocess.run(
            [sys.executable, str(RUNNER), "--config", str(config_path), "--method", method, "--dry-run"],
            cwd=REPO_DIR,
            check=True,
        )
print("Dry-run validation passed for every selected task/regime/method.")

## 5. Run methods

Jobs are launched in pairs so each process owns one T4. RIFT and `spectral_filter` are slower because they run a calibration-gradient pass with temporary rank-one component hooks.

In [ ]:
jobs = [(task, regime, method, seed) for (task, regime) in matrix_configs for seed in seeds for method in methods]

def launch_job(task, regime, method, seed, gpu):
    variant = f"{task}_{regime}_{method}"
    log_path = LOG_DIR / f"{variant}_seed{seed}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu),
        "PYTHONUNBUFFERED": "1",
        "TOKENIZERS_PARALLELISM": "false",
    })
    command = [
        sys.executable, str(RUNNER),
        "--config", str(matrix_configs[(task, regime)]),
        "--method", method,
        "--variant", variant,
        "--seed", str(seed),
        "--output-dir", str(RESULT_DIR),
    ]
    process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=log_handle, stderr=subprocess.STDOUT)
    return process, log_handle, log_path

started = time.perf_counter()
for wave_start in range(0, len(jobs), 2):
    wave = jobs[wave_start:wave_start + 2]
    running = [launch_job(task, regime, method, seed, gpu) for gpu, (task, regime, method, seed) in enumerate(wave)]
    print("Started:", wave)
    while any(process.poll() is None for process, _, _ in running):
        time.sleep(30)
        active = [item for item, run in zip(wave, running) if run[0].poll() is None]
        print("  still running:", active)
    for (task, regime, method, seed), (process, handle, log_path) in zip(wave, running):
        handle.close()
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-30:]
        print(f"\n--- {task}/{regime}/{method} seed={seed}, exit={process.returncode} ---")
        print("\n".join(tail))
        if process.returncode != 0:
            raise RuntimeError(f"{task}/{regime}/{method} seed={seed} failed; inspect {log_path}")

wall_minutes = (time.perf_counter() - started) / 60
print(f"All {len(jobs)} jobs completed in {wall_minutes:.1f} minutes.")

## 6. Analyze and package results

In [ ]:
subprocess.run(
    [
        sys.executable, str(ANALYZER),
        "--input-dir", str(RESULT_DIR),
        "--output-dir", str(ANALYSIS_DIR),
        "--target", "rift",
    ],
    cwd=REPO_DIR,
    check=True,
)
report = (ANALYSIS_DIR / "competitor_report.md").read_text(encoding="utf-8")
print(report)

archive_base = WORK_ROOT / "rift-3b-competitor-results"
archive_path = shutil.make_archive(str(archive_base), "zip", RESULT_DIR)
print("Archive:", archive_path)

## 7. Interpretation rule

Use this notebook to decide whether RIFT survives a 3B task/regime matrix under the same async trace. A positive result supports the thesis direction only if RIFT is competitive on `Accuracy`/`Loss` and improves `Harmful`/`Late harmful` in the hard non-IID high-staleness slices. A negative result is not automatically NO-GO for the small-model thesis, but it means the scale claim must be narrowed until calibration size, loss, and runtime are debugged.